# Chapter 15 &mdash; $A_{TM}$ is Undecidable: the Diagonalization Proof

**Concept 5 of the Chapter 15 decomposition:** *$A_{TM}$ is Undecidable: the Diagonalization Proof*

Assume decider $A$; build $D$ that flips $A(\langle M,M\rangle)$; then ask about $D(\langle D\rangle)$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-A-TM-Diagonalization/Concept-A-TM-Diagonalization.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$$A_{TM} = \{\langle M,w\rangle : M \text{ accepts } w\}$$

**Suppose** $A$ decides $A_{TM}$. Build $D$:

> on input $\langle M\rangle$: run $A$ on $\langle M,M\rangle$; **accept if $A$
> rejects**, **reject if $A$ accepts**.

$D$ is a perfectly good machine: $A$ halts always, so $D$ does too. Now run $D$ on its
own description:

$$D \text{ accepts } \langle D\rangle \iff A \text{ rejects } \langle D,D\rangle
\iff D \text{ does not accept } \langle D\rangle.$$

A contradiction, so $A$ does not exist. The only assumption was $A$'s existence, so
**$A_{TM}$ is undecidable.**

The engine is **diagonalization** &mdash; Cantor's argument, applied to a table of
machines against their own descriptions.

## 2. Definitions

### The diagonal argument, on a finite table

In [ ]:
# Machines as Python predicates; "descriptions" as their names.
MACHINES = {
 'M1': lambda w: w.startswith('1'),
 'M2': lambda w: w.count('0') % 2 == 0,
 'M3': lambda w: len(w) > 2,
 'M4': lambda w: 'M' in w,
 'M5': lambda w: w == w[::-1],
}

def table():
    names = sorted(MACHINES)
    print("      " + "  ".join("%-4s" % n for n in names))
    for n in names:
        row = ["%-4s" % ('acc' if MACHINES[m](n) else '---') for m in names]
        print("%-5s " % n + "  ".join(row))
    return names

### Build the flipped diagonal

In [ ]:
def diagonal_machine():
    names = sorted(MACHINES)
    # D(<M>) accepts iff M does NOT accept <M>
    return lambda w: (w in names) and (not MACHINES[w](w))

## 3. Tests

The table of machines against their own descriptions.

In [ ]:
names = table()
print("\nthe DIAGONAL is M(<M>) for each M:")
for n in names:
    print("   %s(<%s>) = %s" % (n, n, MACHINES[n](n)))

$D$ flips the diagonal, so it differs from **every** listed machine.

In [ ]:
D = diagonal_machine()
for n in names:
    print("   D(<%s>) = %-6s  %s(<%s>) = %-6s  differ? %s"
          % (n, D(n), n, n, MACHINES[n](n), D(n) != MACHINES[n](n)))
    assert D(n) != MACHINES[n](n)
print("\nD disagrees with each machine on that machine's OWN description,")
print("so D is not in the list.")

**The contradiction** arrives when $D$ is asked about itself.

In [ ]:
print("D accepts <D>  iff  A rejects <D,D>")
print("               iff  D does NOT accept <D>")
print()
print("Both branches are impossible, so the assumption fails: A cannot exist.")

Simulating the self-application, to see why it cannot be patched.

In [ ]:
MACHINES['D'] = None          # we would have to fill this in with D itself
def self_apply():
    # D('D') = not MACHINES['D']('D') = not D('D')
    return "D('D') = not D('D')  --  no value satisfies this"
print(self_apply())
del MACHINES['D']
print()
print("It is not that D is hard to compute.  It is that no BOOLEAN works.")

The shape of the argument, for reuse.

In [ ]:
STEPS = [("1. assume", "a decider A for the problem exists"),
         ("2. build",  "a machine D that flips A's answer on the diagonal"),
         ("3. note",   "D is computable BECAUSE A always halts"),
         ("4. ask",    "what does D do on its own description?"),
         ("5. conclude", "both answers contradict, so A does not exist")]
for a, b in STEPS: print("  %-12s %s" % (a, b))
print()
print("Step 3 is the one students skip -- and it is where the proof lives.")

## 4. Exercises


1. Why is step 3 essential? What if $A$ were only a semi-decider?
2. Run the finite table argument with your own five machines.
3. Where does Cantor's uncountability proof use the same flip?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter15/Concept-A-TM-Diagonalization')